In [1]:
using Plots
using Distributions, Random, Statistics
using LinearAlgebra, SparseArrays, JSON
using Colors
using CSV
using DataFrames
using MLJ
using Dates
using JuMP, Ipopt
using LinearAlgebra

purple = RGB(122/256, 30/256, 71/256)
blue   = RGB(0/256, 39/256, 76/256)
mygreen = RGB(83/256, 165/256, 132/256)

In [ ]:
# 1.
# Load Data

dfs = [CSV.read("location$i.csv", DataFrame; types = Dict(:Time => String)) for i in 1:4]

# Convert timestamps and assign location IDs
for (i, data) in enumerate(dfs)
    data.Time = DateTime.(data.Time, dateformat"yyyy-mm-dd HH:MM:SS")
    data.LocationID = fill(i, nrow(data))
end

# Combine all locations and sort by location and time
df = vcat(dfs...)
sort!(df, [:LocationID, :Time])

m     = nrow(df)
times = df.Time
t     = Float64.(df.Power)

location_ids = df.LocationID

# ---------- Normalize weather features ----------

x10m  = Float64.(df.windspeed_10m) ./ maximum(df.windspeed_10m)
x100m = Float64.(df.windspeed_100m) ./ maximum(df.windspeed_100m)
x10g  = Float64.(df.windgusts_10m) ./ maximum(df.windgusts_10m)

# ---------- 24-hour groups, separated by location ----------

hours_per_group = 24

groups = Vector{Int}[]

for loc in unique(location_ids)

    loc_idx = findall(==(loc), location_ids)

    for i in 1:hours_per_group:(length(loc_idx) - hours_per_group + 1)

        push!(groups, loc_idx[i:i + hours_per_group - 1])

    end
end

num_groups = length(groups)

# Partition groups chronologically
train, val, test = partition(1:num_groups, 0.8, 0.1;shuffle = false,rng = 120)

# Convert group selections to observation indices
train_idx = vcat(groups[train]...)
val_idx   = vcat(groups[val]...)
test_idx  = vcat(groups[test]...)

# Exclude observations without a valid previous-hour measurement
train_idx_fit = filter(i -> i > 1 && location_ids[i] == location_ids[i-1] &&times[i] - times[i-1] == Hour(1),train_idx);

In [5]:
# 2.
# Define RBF functions

c = collect(0.0:0.01:1)
num_centers = length(c)

function rbf_features(xs, γ)

    Φ = zeros(length(xs), num_centers)

    for i in eachindex(xs), j in 1:num_centers
        Φ[i, j] = exp(-(γ * abs(xs[i] - c[j]))^2)
    end

    return Φ
end

weather_design_matrix(x1, x2, x3, γ) = hcat(rbf_features(x1, γ),rbf_features(x2, γ),rbf_features(x3, γ))
grouped_design_matrix(x1, x2, x3, lag_power, γ) = hcat(weather_design_matrix(x1, x2, x3, γ),lag_power)

Y_all = t;

In [ ]:
# 3. 
# Training loop

include("market_functions.jl")

function train_rbf(Φ, t; α = 0.01, n_epochs = 15, λ = 0.001,reg_type = :l2, seed = 128)

    rng = MersenneTwister(seed)

    ntrain = size(Φ, 1)
    nfeat  = size(Φ, 2)

    w_0 = 0.0
    w   = zeros(nfeat)

    for epoch in 1:n_epochs

        for i in shuffle(rng, 1:ntrain)

            P_pred = w_0 + dot(Phi[i, :], w)

            ϕi = @view Φ[i, :]

            err = (w_0 + dot(ϕi, w)) - t[i]

            w_0 -= α * err
            w   .-= α * err .* ϕi

            if reg_type == :l2
                w .-= α * λ .* w

            elseif reg_type == :l1
                w .= sign.(w) .* max.(0, abs.(w) .- α * λ)

            end
        end
    end

    return w_0, w
end

predict_rbf(Φ, w_0, w) = w_0 .+ Φ * w